# 04. 1차 EDA 와 통계 검정 — 13클래스 통합검정 vs 정상 대비 이진 재검정

이 프로젝트 통계 설계의 핵심이다.

**13클래스를 한 번에 검정하면 정상 118,800건이 표본을 지배해 효과크기가 희석된다.** 실제로 `Distance` 는 13클래스 Kruskal-Wallis 에서 ε²=0.0023(negligible)이지만, 정상 대비 `a` 유형 1:1 재검정에서는 Cliff's δ=0.95(large, 전체 최댓값)다.

→ 통합검정 단독 결과로 변수를 버리면 안 되고, **정상 대비 이진 재검정을 반드시 병행**해야 한다.

> ⚠️ **1차 EDA 는 `df_tr` 에서만 본다.** valid 를 보고 피처를 설계하면 그 자체가 누수다.

> **출처** — `원본/FDS_전처리_정리본.ipynb` (109셀 · Colab 실행본)
>
> 원본은 한 노트북에 §0~§15 를 전부 담고 있어 어디까지가 한 덩어리인지 알기 어려웠다.
> 이 저장소는 **절 경계 그대로** 5개로 나누고 **원본 실행 출력 98건을 모두 보존**했다.
> 코드는 손대지 않았다 — 셀 순서·내용 모두 원본과 동일하다.



### 재현 조건

| 항목 | 값 |
|---|---|
| 입력 | `train_final.csv` (원본 120,000행 × 64컬럼) |
| 원본 실행 경로 | `/content/train_final.csv` (Google Colab) |
| 저장소 경로 | `data/train.csv` — 용량(54MB) 때문에 미포함, `data/README.md` 참고 |
| 최종 산출물 | `X_tr/X_va/y_tr/y_va.parquet` (3차 전처리 · 58피처) + `label_encoders.pkl` · `le_target.pkl` |

> 노트북을 **순서대로(01→05)** 실행해야 한다. 앞 노트북의 `train` · `df` · `tr_idx`/`va_idx` 를
> 뒤 노트북이 이어받는 구조라, 단독 실행하면 `NameError` 가 난다.
> 각 노트북 첫 셀에 이어받는 변수를 명시해 두었다.

**변수 인계** — 03 에서 `df_tr`, `df_va`, `tr_idx`, `va_idx` 를 이어받는다.

> 원본 셀 범위: `[41] ~ [74]` (총 34셀)


## 10 .1차 EDA 사기 변별 관점 탐색 — `df_tr` 기준

파생변수를 만들기 전에 **어떤 변수가 사기를 가르는지** 확인해 파생 방향을 정한다.
- 반드시 `df_tr` 에서만 본다 (valid를 보고 설계하면 누수).
- 범주형이 아직 라벨 상태라 의미를 그대로 읽을 수 있다.

In [ ]:
# 1차 EDA는 df_tr 에서만 — valid를 보고 설계하면 누수
eda = df_tr.copy()
eda["is_fraud"] = (eda["Fraud_Type"] != "m").astype(int)

num_cols_eda = [c for c in [
    "Transaction_Amount_abs", "Account_balance", "Account_amount_daily_limit",
    "Account_one_month_max_amount", "Distance", "Time_difference_seconds",
    "Number_of_transaction_with_the_account", "Transaction_history_with_the_account",
] if c in eda.columns]

In [ ]:
# 0/1 플래그의 정상 vs 사기 발생률 차이 (클수록 변별력 있는 위협 신호)
flag_cols = [c for c in eda.columns
             if c != "is_fraud" and eda[c].dropna().isin([0, 1]).all() and eda[c].nunique() <= 2]
rate = eda.groupby("is_fraud")[flag_cols].mean().T
rate.columns = ["정상(m)", "사기"]
rate["차이(사기-정상)"] = rate["사기"] - rate["정상(m)"]
display(rate.sort_values("차이(사기-정상)", ascending=False).round(4))

,정상(m),사기,차이(사기-정상)
Transaction_is_withdrawal,0.2925,0.9469,0.6544
Flag_deposit_more_than_tenMillion,0.4238,0.5658,0.1420
Account_release_suspention,0.6399,0.6890,0.0491
Customer_rooting_jailbreak_indicator,0.0728,0.1178,0.0451
Customer_flag_terminal_malicious_behavior_6,0.1427,0.1794,0.0368
Customer_flag_terminal_malicious_behavior_2,0.0961,0.1210,0.0250
Transaction_Failure_Status,0.0234,0.0403,0.0169
Customer_VPN_Indicator,0.0724,0.0849,0.0125
Account_indicator_release_limit_excess,0.2017,0.2134,0.0117
Recipient_account_suspend_status,0.4924,0.4979,0.0055


In [ ]:
#withdrawal,deposit 같이
target = "Fraud_Type"
deposit = "Flag_deposit_more_than_tenMillion"
withdrawal = "Transaction_is_withdrawal"

combo_by_class = (
    eda.groupby([target, withdrawal, deposit])
      .size()
      .reset_index(name="n")
)

combo_by_class["class_total"] = combo_by_class.groupby(target)["n"].transform("sum")
combo_by_class["rate_in_class"] = combo_by_class["n"] / combo_by_class["class_total"]

combo_by_class.sort_values([target, "rate_in_class"], ascending=[True, False])

,Fraud_Type,Transaction_is_withdrawal,Flag_deposit_more_than_tenMillion,n,class_total,rate_in_class
0,a,0,0,29,77,0.376623
1,a,0,1,21,77,0.272727
2,a,1,0,14,77,0.181818
3,a,1,1,13,77,0.168831
4,b,1,0,50,78,0.641026
5,b,1,1,28,78,0.358974
6,c,1,0,40,73,0.547945
7,c,1,1,33,73,0.452055
8,d,1,0,57,84,0.678571
9,d,1,1,27,84,0.321429


In [ ]:
# 범주형이 유형별로 갈리는지 (행 정규화, 사기 유형만)
for c in ['Customer_Gender', 'Customer_credit_rating', 'Customer_loan_type', 'Account_account_type', 'Channel', 'Operating_System', 'Error_Code', 'Type_General_Automatic', 'Access_Medium', 'Location_region']:
    if c in eda.columns:
        print(f"\n=== Fraud_Type × {c} (행 정규화, 사기 유형만) ===")
        ct = pd.crosstab(eda["Fraud_Type"], eda[c], normalize="index")
        display(ct.loc[ct.index != "m"].round(3))



=== Fraud_Type × Customer_Gender (행 정규화, 사기 유형만) ===


Customer_Gender,female,male
Fraud_Type,,
a,0.519,0.481
b,0.577,0.423
c,0.438,0.562
d,0.583,0.417
e,0.519,0.481
f,0.619,0.381
g,0.500,0.500
h,0.488,0.512
i,0.623,0.377



=== Fraud_Type × Customer_credit_rating (행 정규화, 사기 유형만) ===


Customer_credit_rating,A,B,C,D,E,S
Fraud_Type,,,,,,
a,0.208,0.416,0.247,0.013,0.026,0.091
b,0.000,0.000,0.564,0.205,0.231,0.000
c,0.192,0.411,0.205,0.096,0.000,0.096
d,0.107,0.524,0.238,0.048,0.048,0.036
e,0.260,0.442,0.169,0.026,0.052,0.052
f,0.238,0.417,0.238,0.024,0.048,0.036
g,0.292,0.431,0.167,0.042,0.056,0.014
h,0.162,0.588,0.175,0.038,0.012,0.025
i,0.234,0.455,0.156,0.091,0.026,0.039



=== Fraud_Type × Customer_loan_type (행 정규화, 사기 유형만) ===


...

Type_General_Automatic,automatic,general
Fraud_Type,,
a,0.052,0.948
b,0.000,1.000
c,0.000,1.000
d,0.000,1.000
e,0.000,1.000
f,0.000,1.000
g,0.000,1.000
h,0.000,1.000
i,0.000,1.000



=== Fraud_Type × Access_Medium (행 정규화, 사기 유형만) ===


Access_Medium,a,b,c,d,e,f,g
Fraud_Type,,,,,,,
a,0.403,0.494,0.000,0.013,0.039,0.026,0.026
b,0.538,0.462,0.000,0.000,0.000,0.000,0.000
c,0.356,0.493,0.014,0.027,0.055,0.014,0.041
d,0.536,0.464,0.000,0.000,0.000,0.000,0.000
e,0.299,0.455,0.026,0.065,0.065,0.078,0.013
f,0.369,0.452,0.024,0.036,0.048,0.024,0.048
g,0.347,0.472,0.042,0.042,0.056,0.042,0.000
h,0.450,0.388,0.025,0.000,0.062,0.038,0.038
i,0.325,0.545,0.000,0.000,0.078,0.039,0.013



=== Fraud_Type × Location_region (행 정규화, 사기 유형만) ===


Location_region,강원도,경기도,경상남도,경상북도,광주광역시,대구광역시,대전광역시,부산광역시,서울특별시,세종특별자치시,울산광역시,인천광역시,전라남도,전라북도,제주특별자치도,충청남도,충청북도
Fraud_Type,,,,,,,,,,,,,,,,,
a,0.208,0.117,0.078,0.026,0.026,0.000,0.013,0.039,0.091,0.000,0.052,0.039,0.195,0.013,0.052,0.013,0.039
b,0.051,0.128,0.103,0.115,0.026,0.013,0.000,0.026,0.038,0.000,0.026,0.026,0.128,0.103,0.000,0.090,0.128
c,0.068,0.110,0.110,0.151,0.000,0.000,0.027,0.041,0.041,0.014,0.000,0.000,0.096,0.041,0.000,0.178,0.123
d,0.167,0.155,0.119,0.190,0.000,0.012,0.024,0.012,0.024,0.012,0.024,0.012,0.083,0.024,0.000,0.071,0.071
e,0.052,0.156,0.130,0.091,0.013,0.026,0.013,0.026,0.065,0.013,0.000,0.026,0.117,0.039,0.026,0.104,0.104
f,0.143,0.107,0.095,0.107,0.000,0.000,0.012,0.036,0.048,0.012,0.024,0.000,0.071,0.131,0.000,0.107,0.107
g,0.111,0.167,0.056,0.153,0.028,0.000,0.014,0.014,0.014,0.014,0.000,0.000,0.083,0.042,0.000,0.139,0.167
h,0.150,0.075,0.062,0.088,0.012,0.012,0.012,0.000,0.025,0.012,0.025,0.038,0.100,0.050,0.025,0.125,0.188
i,0.039,0.130,0.169,0.078,0.026,0.000,0.013,0.026,0.013,0.000,0.013,0.039,0.143,0.078,0.000,0.104,0.130


### 10-4. Fraud_Type별 Location_region 비율표

In [ ]:
# 1. Fraud_Type별 Location_region 비율표
location_region_ratio = pd.crosstab(
    train["Fraud_Type"],
    train["Location_region"],
    normalize="index"
).round(3)

display(location_region_ratio)

# 2. 정상 m 기준 차이 계산
normal_region_ratio = location_region_ratio.loc["m"]

fraud_vs_normal_region = location_region_ratio.drop(index="m").sub(
    normal_region_ratio,
    axis=1
).round(3)

display(fraud_vs_normal_region)

# 3. 유형별 정상 대비 높은 지역 TOP 5
for fraud_type in fraud_vs_normal_region.index:
    print(f"\n=== Fraud_Type {fraud_type}: 정상 m 대비 높은 지역 TOP 5 ===")
    display(
        fraud_vs_normal_region.loc[fraud_type]
        .sort_values(ascending=False)
        .head(5)
    )

Location_region,강원도,경기도,경상남도,경상북도,광주광역시,대구광역시,대전광역시,부산광역시,서울특별시,세종특별자치시,울산광역시,인천광역시,전라남도,전라북도,제주특별자치도,충청남도,충청북도
Fraud_Type,,,,,,,,,,,,,,,,,
a,0.190,0.110,0.090,0.030,0.030,0.000,0.010,0.030,0.070,0.000,0.040,0.04,0.200,0.020,0.070,0.010,0.06
b,0.060,0.150,0.090,0.110,0.020,0.030,0.000,0.020,0.030,0.000,0.020,0.04,0.120,0.080,0.000,0.110,0.12
c,0.060,0.150,0.090,0.140,0.000,0.000,0.020,0.050,0.040,0.010,0.010,0.00,0.100,0.030,0.000,0.190,0.11
d,0.140,0.140,0.110,0.180,0.000,0.020,0.020,0.010,0.020,0.010,0.040,0.01,0.100,0.020,0.000,0.080,0.10
e,0.090,0.130,0.150,0.120,0.010,0.020,0.010,0.020,0.060,0.020,0.000,0.02,0.120,0.050,0.020,0.080,0.08
f,0.130,0.110,0.080,0.130,0.000,0.000,0.010,0.040,0.040,0.010,0.030,0.00,0.090,0.110,0.000,0.110,0.11
g,0.100,0.180,0.050,0.170,0.020,0.020,0.010,0.020,0.010,0.010,0.000,0.01,0.060,0.070,0.000,0.130,0.14
h,0.130,0.120,0.080,0.090,0.010,0.030,0.010,0.000,0.020,0.010,0.020,0.03,0.090,0.040,0.020,0.110,0.19
i,0.040,0.160,0.150,0.100,0.020,0.000,0.020,0.030,0.010,0.000,0.010,0.04,0.120,0.090,0.000,0.090,0.12


Location_region,강원도,경기도,경상남도,경상북도,광주광역시,대구광역시,대전광역시,부산광역시,서울특별시,세종특별자치시,울산광역시,인천광역시,전라남도,전라북도,제주특별자치도,충청남도,충청북도
Fraud_Type,,,,,,,,,,,,,,,,,
a,0.095,-0.044,0.006,-0.092,0.014,-0.023,-0.003,0.012,0.034,-0.011,0.026,0.02,0.099,-0.047,0.065,-0.092,-0.06
b,-0.035,-0.004,0.006,-0.012,0.004,0.007,-0.013,0.002,-0.006,-0.011,0.006,0.02,0.019,0.013,-0.005,0.008,0.00
c,-0.035,-0.004,0.006,0.018,-0.016,-0.023,0.007,0.032,0.004,-0.001,-0.004,-0.02,-0.001,-0.037,-0.005,0.088,-0.01
d,0.045,-0.014,0.026,0.058,-0.016,-0.003,0.007,-0.008,-0.016,-0.001,0.026,-0.01,-0.001,-0.047,-0.005,-0.022,-0.02
e,-0.005,-0.024,0.066,-0.002,-0.006,-0.003,-0.003,0.002,0.024,0.009,-0.014,0.00,0.019,-0.017,0.015,-0.022,-0.04
f,0.035,-0.044,-0.004,0.008,-0.016,-0.023,-0.003,0.022,0.004,-0.001,0.016,-0.02,-0.011,0.043,-0.005,0.008,-0.01
g,0.005,0.026,-0.034,0.048,0.004,-0.003,-0.003,0.002,-0.026,-0.001,-0.014,-0.01,-0.041,0.003,-0.005,0.028,0.02
h,0.035,-0.034,-0.004,-0.032,-0.006,0.007,-0.003,-0.018,-0.016,-0.001,0.006,0.01,-0.011,-0.027,0.015,0.008,0.07
i,-0.055,0.006,0.066,-0.022,0.004,-0.023,0.007,0.012,-0.026,-0.011,-0.004,0.02,0.019,0.023,-0.005,-0.012,0.00



=== Fraud_Type a: 정상 m 대비 높은 지역 TOP 5 ===


,a
Location_region,
전라남도,0.099
강원도,0.095
제주특별자치도,0.065
서울특별시,0.034
울산광역시,0.026



=== Fraud_Type b: 정상 m 대비 높은 지역 TOP 5 ===


...

,j
Location_region,
전라남도,0.049
경상남도,0.036
강원도,0.025
전라북도,0.023
인천광역시,0.010



=== Fraud_Type k: 정상 m 대비 높은 지역 TOP 5 ===


,k
Location_region,
서울특별시,0.044
울산광역시,0.026
강원도,0.025
전라북도,0.013
부산광역시,0.012



=== Fraud_Type l: 정상 m 대비 높은 지역 TOP 5 ===


,l
Location_region,
경상북도,0.028
서울특별시,0.024
충청북도,0.020
충청남도,0.018
제주특별자치도,0.015


### 10-5. 수치형 변수 Kruskal-Wallis 스크리닝 (13그룹 동시비교, 효과크기 희석 주의)

In [ ]:
#수치형 크루스칼 검정, 효과크기
import numpy as np
import pandas as pd
from scipy.stats import kruskal

num_cols = [
    'Customer_Birthyear',
    'Account_initial_balance',
    'Account_balance',
    'Account_amount_daily_limit',
    'Account_remaining_amount_daily_limit_exceeded',
    'Account_one_month_max_amount',
    'Account_one_month_std_dev',
    'Account_dawn_one_month_max_amount',
    'Account_dawn_one_month_std_dev',
    'Transaction_num_connection_failure',
    'Distance',
    'Number_of_transaction_with_the_account',
    'Transaction_history_with_the_account',
    'Time_difference_seconds',
    'Transaction_Amount_abs'
]

target_col = 'Fraud_Type'

def epsilon_squared_kruskal(h_stat, n, k):
    """
    Kruskal-Wallis effect size: epsilon-squared.
    값이 클수록 Fraud_Type별 분포 차이가 큼.
    """
    if n <= k:
        return np.nan
    return (h_stat - k + 1) / (n - k)

def effect_size_label(eps):
    """
    일반적인 참고 기준.
    맥락에 따라 절대 기준으로 과해석하지 않는 것이 좋음.
    """
    if pd.isna(eps):
        return 'nan'
    elif eps < 0.01:
        return 'negligible'
    elif eps < 0.06:
        return 'small'
    elif eps < 0.14:
        return 'medium'
    else:
        return 'large'

results = []

for col in num_cols:
    temp = train[[target_col, col]].dropna()

    groups = [
        group[col].values
        for _, group in temp.groupby(target_col)
    ]

    k = len(groups)
    n = len(temp)

    if k < 2:
        h_stat = np.nan
        p_value = np.nan
        eps_sq = np.nan
    else:
        h_stat, p_value = kruskal(*groups)
        eps_sq = epsilon_squared_kruskal(h_stat, n, k)

    results.append({
        'feature': col,
        'kruskal_h_stat': h_stat,
        'kruskal_p_value': p_value,
        'epsilon_squared': eps_sq,
        'effect_size': effect_size_label(eps_sq),
        'n_samples': n,
        'n_groups': k
    })

kruskal_result = (
    pd.DataFrame(results)
    .sort_values('epsilon_squared', ascending=False)
    .reset_index(drop=True)
)

kruskal_result

,feature,kruskal_h_stat,kruskal_p_value,epsilon_squared,effect_size,n_samples,n_groups
0,Transaction_Amount_abs,653.590223,3.745918e-132,0.005347,negligible,120000,13
1,Account_balance,495.988893,1.582042e-98,0.004034,negligible,120000,13
2,Number_of_transaction_with_the_account,394.991157,4.349550e-77,0.003192,negligible,120000,13
3,Distance,278.606828,1.437738e-52,0.002222,negligible,120000,13
4,Transaction_history_with_the_account,277.269082,2.740267e-52,0.002211,negligible,120000,13
5,Account_one_month_std_dev,226.941140,8.609595e-42,0.001791,negligible,120000,13
6,Account_one_month_max_amount,217.801244,6.780261e-40,0.001715,negligible,120000,13
7,Time_difference_seconds,188.873204,6.405975e-34,0.001476,negligible,119844,13
8,Customer_Birthyear,145.791385,4.041421e-25,0.001115,negligible,120000,13
9,Account_dawn_one_month_max_amount,133.252792,1.370570e-22,0.001011,negligible,120000,13


# 11. 통계검정 — 유형별 vs 정상 개별비교 (Mann-Whitney/카이제곱 + 효과크기)

### 11-1. 범주형 변수 카이제곱 검정 함수

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

cat_cols = [
    'Customer_Gender',
    'Customer_credit_rating',
    'Customer_loan_type',
    'Account_account_type',
    'Channel',
    'Operating_System',
    'Error_Code',
    'Type_General_Automatic',
    'Access_Medium',
    'Location_region'
]

target_col = 'Fraud_Type'

def cramers_v_corrected(confusion_matrix):
    """
    Bias-corrected Cramér's V.
    범주형 변수와 범주형 타겟 간 효과크기.
    0에 가까우면 약함, 1에 가까우면 강함.
    """
    chi2, _, _, _ = chi2_contingency(confusion_matrix)

    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan

    r, k = confusion_matrix.shape
    phi2 = chi2 / n

    # Bias correction
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    denom = min(k_corr - 1, r_corr - 1)
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2_corr / denom)

def cramers_v_label(v):
    """
    참고용 기준.
    범주 수와 도메인에 따라 절대 기준으로 과해석하지 않는 것이 좋음.
    """
    if pd.isna(v):
        return 'nan'
    elif v < 0.10:
        return 'negligible'
    elif v < 0.30:
        return 'small'
    elif v < 0.50:
        return 'medium'
    else:
        return 'large'

results = []

for col in cat_cols:
    temp = train[[target_col, col]].dropna().copy()

    contingency = pd.crosstab(temp[col], temp[target_col])

    # 행 또는 열이 1개뿐이면 독립성 검정 불가
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        chi2_stat = np.nan
        p_value = np.nan
        dof = np.nan
        cramers_v = np.nan
    else:
        chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
        cramers_v = cramers_v_corrected(contingency)

    results.append({
        'feature': col,
        'chi2_stat': chi2_stat,
        'chi2_p_value': p_value,
        'dof': dof,
        'cramers_v': cramers_v,
        'effect_size': cramers_v_label(cramers_v),
        'n_samples': len(temp),
        'n_categories': contingency.shape[0],
        'min_expected_freq': np.nan if contingency.shape[0] < 2 else expected.min()
    })

cat_target_result = (
    pd.DataFrame(results)
    .sort_values('cramers_v', ascending=False)
    .reset_index(drop=True)
)

cat_target_result

,feature,chi2_stat,chi2_p_value,dof,cramers_v,effect_size,n_samples,n_categories,min_expected_freq
0,Channel,822.951511,1.619610e-149,36,0.046755,negligible,120000,4,24.337500
1,Error_Code,112.270085,2.126028e-18,12,0.028907,negligible,120000,2,2.343333
2,Type_General_Automatic,71.927308,1.393879e-10,12,0.022347,negligible,120000,2,6.450000
3,Customer_credit_rating,301.312079,7.550989e-34,60,0.020055,negligible,120000,6,4.957500
4,Account_account_type,105.301056,1.025477e-08,36,0.013875,negligible,120000,4,22.775833
5,Operating_System,156.513958,1.497091e-10,60,0.012683,negligible,120000,6,6.064167
6,Location_region,317.819534,2.936542e-08,192,0.009348,negligible,120000,17,0.515833
7,Access_Medium,131.125291,2.585180e-05,72,0.009062,negligible,120000,7,2.615000
8,Customer_loan_type,85.370229,7.259672e-04,48,0.008824,negligible,120000,5,4.951667
9,Customer_Gender,6.178158,9.068360e-01,12,0.000000,negligible,120000,2,48.951667


### 11-2. 정상(m) vs 사기 전체(a~l) 이진화 데이터셋 구성

In [ ]:
#2. 정상 m vs 사기 전체 a~l
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

target_col = 'Fraud_Type'
normal_label = 'm'

num_cols = [
    'Customer_Birthyear',
    'Account_initial_balance',
    'Account_balance',
    'Account_amount_daily_limit',
    'Account_remaining_amount_daily_limit_exceeded',
    'Account_one_month_max_amount',
    'Account_one_month_std_dev',
    'Account_dawn_one_month_max_amount',
    'Account_dawn_one_month_std_dev',
    'Transaction_num_connection_failure',
    'Distance',
    'Number_of_transaction_with_the_account',
    'Transaction_history_with_the_account',
    'Time_difference_seconds',
    'Transaction_Amount_abs'
]

cat_cols = [
    'Customer_Gender',
    'Customer_credit_rating',
    'Customer_loan_type',
    'Account_account_type',
    'Channel',
    'Operating_System',
    'Error_Code',
    'Type_General_Automatic',
    'Access_Medium',
    'Location_region'
]

train_binary = train.copy()
train_binary['is_fraud'] = np.where(train_binary[target_col] == normal_label, 0, 1)

### 11-3. 효과크기 함수 — Cliff's Delta

In [ ]:
#수치형: Mann-Whitney U + Cliff’s delta
def cliffs_delta(x, y):
    """
    x: fraud group
    y: normal group

    delta > 0: 사기 그룹 값이 정상보다 큰 경향
    delta < 0: 사기 그룹 값이 정상보다 작은 경향
    """
    x = pd.Series(x).dropna().to_numpy()
    y = pd.Series(y).dropna().to_numpy()

    n_x = len(x)
    n_y = len(y)

    if n_x == 0 or n_y == 0:
        return np.nan

    u_stat, _ = mannwhitneyu(x, y, alternative='two-sided')
    return (2 * u_stat) / (n_x * n_y) - 1

def cliffs_label(delta):
    abs_delta = abs(delta)

    if pd.isna(delta):
        return 'nan'
    elif abs_delta < 0.147:
        return 'negligible'
    elif abs_delta < 0.33:
        return 'small'
    elif abs_delta < 0.474:
        return 'medium'
    else:
        return 'large'

num_results = []

fraud_df = train_binary[train_binary['is_fraud'] == 1]
normal_df = train_binary[train_binary['is_fraud'] == 0]

for col in num_cols:
    fraud_values = fraud_df[col].dropna()
    normal_values = normal_df[col].dropna()

    if len(fraud_values) == 0 or len(normal_values) == 0:
        u_stat = np.nan
        p_value = np.nan
        delta = np.nan
    else:
        u_stat, p_value = mannwhitneyu(
            fraud_values,
            normal_values,
            alternative='two-sided'
        )
        delta = cliffs_delta(fraud_values, normal_values)

    num_results.append({
        'feature': col,
        'mannwhitney_u_stat': u_stat,
        'mannwhitney_p_value': p_value,
        'cliffs_delta': delta,
        'abs_cliffs_delta': abs(delta),
        'effect_size': cliffs_label(delta),
        'fraud_median': fraud_values.median(),
        'normal_median': normal_values.median(),
        'direction': (
            'fraud_higher' if delta > 0
            else 'fraud_lower' if delta < 0
            else 'same'
        ),
        'fraud_n': len(fraud_values),
        'normal_n': len(normal_values)
    })

num_binary_result = (
    pd.DataFrame(num_results)
    .sort_values('abs_cliffs_delta', ascending=False)
    .reset_index(drop=True)
)

num_binary_result

,feature,mannwhitney_u_stat,mannwhitney_p_value,cliffs_delta,abs_cliffs_delta,effect_size,fraud_median,normal_median,direction,fraud_n,normal_n
0,Transaction_Amount_abs,99407163.0,1.010641e-122,0.394601,0.394601,medium,9.895000e+06,2.650000e+06,fraud_higher,1200,118800
1,Account_balance,45555712.0,5.922393e-103,-0.360891,0.360891,medium,6.930936e+06,1.106274e+07,fraud_lower,1200,118800
2,Time_difference_seconds,63619770.5,4.046552e-10,-0.104816,0.104816,negligible,7.689000e+03,9.197000e+03,fraud_lower,1198,118646
3,Distance,78111673.0,1.054528e-08,0.095843,0.095843,negligible,1.712611e+02,1.556638e+02,fraud_higher,1200,118800
4,Account_one_month_max_amount,77202201.5,6.976027e-07,0.083084,0.083084,negligible,1.647000e+07,1.414000e+07,fraud_higher,1200,118800
5,Transaction_num_connection_failure,76313030.5,1.465645e-06,0.070609,0.070609,negligible,0.000000e+00,0.000000e+00,fraud_higher,1200,118800
6,Number_of_transaction_with_the_account,75757658.5,1.135054e-05,0.062818,0.062818,negligible,0.000000e+00,0.000000e+00,fraud_higher,1200,118800
7,Account_initial_balance,75473765.5,4.440863e-04,0.058835,0.058835,negligible,1.018078e+07,9.680268e+06,fraud_higher,1200,118800
8,Customer_Birthyear,67912911.0,4.794697e-03,-0.047238,0.047238,negligible,1.975000e+03,1.977000e+03,fraud_lower,1200,118800
9,Account_dawn_one_month_std_dev,68907971.5,2.729614e-02,-0.033278,0.033278,negligible,0.000000e+00,0.000000e+00,fraud_lower,1200,118800


### 11-4. 효과크기 함수 — Cramér's V + Odds Ratio

In [ ]:
#범주형: Chi-square + Cramér’s V + Odds Ratio
def cramers_v_corrected(confusion_matrix):
    chi2, _, _, _ = chi2_contingency(confusion_matrix)

    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan

    r, k = confusion_matrix.shape
    phi2 = chi2 / n

    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    denom = min(k_corr - 1, r_corr - 1)
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2_corr / denom)

def cramers_label(v):
    if pd.isna(v):
        return 'nan'
    elif v < 0.10:
        return 'negligible'
    elif v < 0.30:
        return 'small'
    elif v < 0.50:
        return 'medium'
    else:
        return 'large'

def odds_ratio_for_binary_table(contingency):
    """
    2x2 테이블에서 odds ratio 계산.
    행: 범주형 변수 값 2개
    열: is_fraud 0/1

    Haldane-Anscombe correction 적용.
    """
    table = contingency.copy().astype(float)

    if table.shape != (2, 2):
        return np.nan

    table = table + 0.5

    a = table.iloc[0, 0]
    b = table.iloc[0, 1]
    c = table.iloc[1, 0]
    d = table.iloc[1, 1]

    return (d / c) / (b / a)

cat_results = []

for col in cat_cols:
    temp = train_binary[[col, 'is_fraud']].dropna().copy()

    contingency = pd.crosstab(temp[col], temp['is_fraud'])

    # 혹시 한쪽 클래스가 빠진 경우를 대비해 0/1 열 보정
    for class_value in [0, 1]:
        if class_value not in contingency.columns:
            contingency[class_value] = 0

    contingency = contingency[[0, 1]]

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        chi2_stat = np.nan
        p_value = np.nan
        dof = np.nan
        cramers_v = np.nan
        min_expected = np.nan
    else:
        chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
        cramers_v = cramers_v_corrected(contingency)
        min_expected = expected.min()

    odds_ratio = (
        odds_ratio_for_binary_table(contingency)
        if contingency.shape == (2, 2)
        else np.nan
    )

    cat_results.append({
        'feature': col,
        'chi2_stat': chi2_stat,
        'chi2_p_value': p_value,
        'dof': dof,
        'cramers_v': cramers_v,
        'effect_size': cramers_label(cramers_v),
        'odds_ratio_if_binary': odds_ratio,
        'n_samples': len(temp),
        'n_categories': contingency.shape[0],
        'min_expected_freq': min_expected
    })

cat_binary_result = (
    pd.DataFrame(cat_results)
    .sort_values('cramers_v', ascending=False)
    .reset_index(drop=True)
)

cat_binary_result

,feature,chi2_stat,chi2_p_value,dof,cramers_v,effect_size,odds_ratio_if_binary,n_samples,n_categories,min_expected_freq
0,Type_General_Automatic,53.451679,2.650323e-13,1,0.020907,negligible,5.319590,120000,2,77.40
1,Account_account_type,48.293440,1.844275e-10,3,0.019428,negligible,NaN,120000,4,273.31
2,Channel,31.757792,5.886122e-07,3,0.015481,negligible,NaN,120000,4,292.05
3,Location_region,27.673903,3.457367e-02,16,0.009863,negligible,NaN,120000,17,6.19
4,Error_Code,12.426208,4.233507e-04,1,0.009758,negligible,1.727799,120000,2,28.12
5,Customer_credit_rating,13.343637,2.036275e-02,5,0.008338,negligible,NaN,120000,6,59.49
6,Customer_loan_type,12.097052,1.664397e-02,4,0.008214,negligible,NaN,120000,5,59.42
7,Customer_Gender,0.118053,7.311556e-01,1,0.000000,negligible,0.978652,120000,2,587.42
8,Operating_System,4.988138,4.173296e-01,5,0.000000,negligible,NaN,120000,6,72.77
9,Access_Medium,5.708301,4.566464e-01,6,0.000000,negligible,NaN,120000,7,31.38


### 11-5. 정상 vs 각 사기 유형(1:1) 비교 준비

In [ ]:
# 정상 m vs 각 사기 유형
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

target_col = 'Fraud_Type'
normal_label = 'm'

num_cols = [
    'Customer_Birthyear',
    'Account_initial_balance',
    'Account_balance',
    'Account_amount_daily_limit',
    'Account_remaining_amount_daily_limit_exceeded',
    'Account_one_month_max_amount',
    'Account_one_month_std_dev',
    'Account_dawn_one_month_max_amount',
    'Account_dawn_one_month_std_dev',
    'Transaction_num_connection_failure',
    'Distance',
    'Number_of_transaction_with_the_account',
    'Transaction_history_with_the_account',
    'Time_difference_seconds',
    'Transaction_Amount_abs'
]

cat_cols = [
    'Customer_Gender',
    'Customer_credit_rating',
    'Customer_loan_type',
    'Account_account_type',
    'Channel',
    'Operating_System',
    'Error_Code',
    'Type_General_Automatic',
    'Access_Medium',
    'Location_region'
]

### 11-6. 공통 함수 재정의

In [ ]:
#공통 함수
def cliffs_delta(x, y):
    x = pd.Series(x).dropna().to_numpy()
    y = pd.Series(y).dropna().to_numpy()

    n_x = len(x)
    n_y = len(y)

    if n_x == 0 or n_y == 0:
        return np.nan

    u_stat, _ = mannwhitneyu(x, y, alternative='two-sided')
    return (2 * u_stat) / (n_x * n_y) - 1

def cliffs_label(delta):
    abs_delta = abs(delta)

    if pd.isna(delta):
        return 'nan'
    elif abs_delta < 0.147:
        return 'negligible'
    elif abs_delta < 0.33:
        return 'small'
    elif abs_delta < 0.474:
        return 'medium'
    else:
        return 'large'

def cramers_v_corrected(confusion_matrix):
    chi2, _, _, _ = chi2_contingency(confusion_matrix)

    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan

    r, k = confusion_matrix.shape
    phi2 = chi2 / n

    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    denom = min(k_corr - 1, r_corr - 1)
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2_corr / denom)

def cramers_label(v):
    if pd.isna(v):
        return 'nan'
    elif v < 0.10:
        return 'negligible'
    elif v < 0.30:
        return 'small'
    elif v < 0.50:
        return 'medium'
    else:
        return 'large'

### 11-7. 수치형 변수 — 유형별 vs 정상 비교

In [ ]:
#수치형 변수 정상 vs 사기
normal_df = train_binary[train_binary[target_col] == normal_label]

fraud_labels = sorted([
    label for label in train_binary[target_col].dropna().unique()
    if label != normal_label
])

num_pair_results = []

for fraud_label in fraud_labels:
    fraud_df = train_binary[train_binary[target_col] == fraud_label]

    for col in num_cols:
        fraud_values = fraud_df[col].dropna()
        normal_values = normal_df[col].dropna()

        if len(fraud_values) == 0 or len(normal_values) == 0:
            u_stat = np.nan
            p_value = np.nan
            delta = np.nan
        else:
            u_stat, p_value = mannwhitneyu(
                fraud_values,
                normal_values,
                alternative='two-sided'
            )
            delta = cliffs_delta(fraud_values, normal_values)

        num_pair_results.append({
            'fraud_type': fraud_label,
            'feature': col,
            'mannwhitney_u_stat': u_stat,
            'mannwhitney_p_value': p_value,
            'cliffs_delta': delta,
            'abs_cliffs_delta': abs(delta),
            'effect_size': cliffs_label(delta),
            'fraud_median': fraud_values.median(),
            'normal_median': normal_values.median(),
            'direction': (
                'fraud_higher' if delta > 0
                else 'fraud_lower' if delta < 0
                else 'same'
            ),
            'fraud_n': len(fraud_values),
            'normal_n': len(normal_values)
        })

num_pair_result = (
    pd.DataFrame(num_pair_results)
    .sort_values(['fraud_type', 'abs_cliffs_delta'], ascending=[True, False])
    .reset_index(drop=True)
)

num_pair_result

,fraud_type,feature,mannwhitney_u_stat,mannwhitney_p_value,cliffs_delta,abs_cliffs_delta,effect_size,fraud_median,normal_median,direction,fraud_n,normal_n
0,a,Distance,11609092.0,2.483786e-61,0.954393,0.954393,large,3.382223e+02,1.556638e+02,fraud_higher,100,118800
1,a,Time_difference_seconds,2089053.5,4.741730e-28,-0.640664,0.640664,large,3.350500e+03,9.197000e+03,fraud_lower,98,118646
2,a,Account_initial_balance,4939618.5,3.547981e-03,-0.168414,0.168414,small,7.135974e+06,9.680268e+06,fraud_lower,100,118800
3,a,Number_of_transaction_with_the_account,6839095.0,2.151554e-03,0.151363,0.151363,small,1.000000e+00,0.000000e+00,fraud_higher,100,118800
4,a,Transaction_Amount_abs,6597963.0,5.512337e-02,0.110768,0.110768,negligible,4.025000e+06,2.650000e+06,fraud_higher,100,118800
...,...,...,...,...,...,...,...,...,...,...,...,...
175,l,Account_dawn_one_month_std_dev,6279035.5,2.723282e-01,0.057077,0.057077,negligible,0.000000e+00,0.000000e+00,fraud_higher,100,118800
176,l,Transaction_history_with_the_account,5607797.0,2.989198e-01,-0.055926,0.055926,negligible,0.000000e+00,1.000000e+00,fraud_lower,100,118800
177,l,Distance,5640277.0,3.823401e-01,-0.050458,0.050458,negligible,1.482231e+02,1.556638e+02,fraud_lower,100,118800
178,l,Number_of_transaction_with_the_account,5768477.0,5.582505e-01,-0.028876,0.028876,negligible,0.000000e+00,0.000000e+00,fraud_lower,100,118800


### 11-8. 유형별 TOP3 수치형 변수 (Cliff's Delta 기준)

In [ ]:
#top3
num_top3_by_type = (
    num_pair_result
    .sort_values(['fraud_type', 'abs_cliffs_delta'], ascending=[True, False])
    .groupby('fraud_type')['feature']
    .apply(lambda x: list(x.head(3)))
    .reset_index(name='num_top3_features')
)

num_top3_by_type

,fraud_type,num_top3_features
0,a,"[Distance, Time_difference_seconds, Account_in..."
1,b,[Account_remaining_amount_daily_limit_exceeded...
2,c,"[Account_balance, Transaction_Amount_abs, Acco..."
3,d,"[Transaction_Amount_abs, Account_balance, Acco..."
4,e,"[Account_balance, Transaction_Amount_abs, Acco..."
5,f,"[Transaction_Amount_abs, Account_balance, Acco..."
6,g,"[Transaction_Amount_abs, Account_balance, Acco..."
7,h,"[Account_one_month_std_dev, Account_one_month_..."
8,i,"[Transaction_Amount_abs, Account_one_month_std..."
9,j,"[Transaction_history_with_the_account, Transac..."


### 11-9. 범주형 변수 — 유형별 vs 정상 비교

In [ ]:
#범주형 변수 정상 vs 사기
cat_pair_results = []

for fraud_label in fraud_labels:
    pair_df = train_binary[train_binary[target_col].isin([normal_label, fraud_label])].copy()
    pair_df['is_target_fraud_type'] = np.where(
        pair_df[target_col] == fraud_label,
        1,
        0
    )

    for col in cat_cols:
        temp = pair_df[[col, 'is_target_fraud_type']].dropna().copy()

        contingency = pd.crosstab(temp[col], temp['is_target_fraud_type'])

        for class_value in [0, 1]:
            if class_value not in contingency.columns:
                contingency[class_value] = 0

        contingency = contingency[[0, 1]]

        if contingency.shape[0] < 2 or contingency.shape[1] < 2:
            chi2_stat = np.nan
            p_value = np.nan
            dof = np.nan
            cramers_v = np.nan
            min_expected = np.nan
        else:
            chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
            cramers_v = cramers_v_corrected(contingency)
            min_expected = expected.min()

        cat_pair_results.append({
            'fraud_type': fraud_label,
            'feature': col,
            'chi2_stat': chi2_stat,
            'chi2_p_value': p_value,
            'dof': dof,
            'cramers_v': cramers_v,
            'effect_size': cramers_label(cramers_v),
            'n_samples': len(temp),
            'n_categories': contingency.shape[0],
            'min_expected_freq': min_expected
        })

cat_pair_result = (
    pd.DataFrame(cat_pair_results)
    .sort_values(['fraud_type', 'cramers_v'], ascending=[True, False])
    .reset_index(drop=True)
)

cat_pair_result

,fraud_type,feature,chi2_stat,chi2_p_value,dof,cramers_v,effect_size,n_samples,n_categories,min_expected_freq
0,a,Location_region,139.654618,8.419083e-22,16,0.032249,negligible,118900,17,0.514718
1,a,Customer_loan_type,5.614650,2.298340e-01,4,0.003685,negligible,118900,5,4.963835
2,a,Customer_credit_rating,6.111422,2.955276e-01,5,0.003057,negligible,118900,6,4.948696
3,a,Customer_Gender,0.258468,6.111743e-01,1,0.000000,negligible,118900,2,48.959630
4,a,Account_account_type,1.686634,6.399094e-01,3,0.000000,negligible,118900,4,22.714886
...,...,...,...,...,...,...,...,...,...,...
115,l,Customer_Gender,0.000000,1.000000e+00,1,0.000000,negligible,118900,2,48.957107
116,l,Channel,2.259139,5.203939e-01,3,0.000000,negligible,118900,4,24.292683
117,l,Operating_System,1.700384,8.888514e-01,5,0.000000,negligible,118900,6,6.074012
118,l,Error_Code,0.000000,1.000000e+00,1,0.000000,negligible,118900,2,2.327166


### 11-10. 유형별 TOP3 범주형 변수 (Cramér's V 기준)

In [ ]:
#top 3
cat_top3_by_type = (
    cat_pair_result
    .sort_values(['fraud_type', 'cramers_v'], ascending=[True, False])
    .groupby('fraud_type')['feature']
    .apply(lambda x: list(x.head(3)))
    .reset_index(name='cat_top3_features')
)

cat_top3_by_type

,fraud_type,cat_top3_features
0,a,"[Location_region, Customer_loan_type, Customer..."
1,b,"[Customer_credit_rating, Channel, Operating_Sy..."
2,c,"[Channel, Location_region, Operating_System]"
3,d,"[Channel, Access_Medium, Account_account_type]"
4,e,"[Channel, Operating_System, Access_Medium]"
5,f,"[Channel, Operating_System, Account_account_type]"
6,g,"[Type_General_Automatic, Customer_credit_ratin..."
7,h,"[Error_Code, Type_General_Automatic, Customer_..."
8,i,"[Type_General_Automatic, Channel, Location_reg..."
9,j,"[Account_account_type, Access_Medium, Error_Code]"


### 11-11. Phi 계수 함수 (2x2 전용)

In [ ]:
def phi_coefficient(table):
    table = table.astype(float)

    a = table.loc[0, 0]
    b = table.loc[0, 1]
    c = table.loc[1, 0]
    d = table.loc[1, 1]

    denom = np.sqrt((a + b) * (c + d) * (a + c) * (b + d))

    if denom == 0:
        return np.nan

    return ((a * d) - (b * c)) / denom

def odds_ratio_flag1(table):
    """
    flag=1일 때 사기 odds / flag=0일 때 사기 odds
    Haldane-Anscombe correction 적용
    """
    table = table.astype(float) + 0.5

    flag0_normal = table.loc[0, 0]
    flag0_fraud = table.loc[0, 1]
    flag1_normal = table.loc[1, 0]
    flag1_fraud = table.loc[1, 1]

    odds_flag0 = flag0_fraud / flag0_normal
    odds_flag1 = flag1_fraud / flag1_normal

    return odds_flag1 / odds_flag0

### 11-12. 이진 플래그 변수 — 카이제곱/Fisher's + Phi + Odds Ratio

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

target_col = 'Fraud_Type'
normal_label = 'm'

flag_cols = [
    'Customer_flag_change_of_authentication_1',
    'Customer_flag_change_of_authentication_2',
    'Customer_flag_change_of_authentication_3',
    'Customer_flag_change_of_authentication_4',
    'Customer_rooting_jailbreak_indicator',
    'Customer_mobile_roaming_indicator',
    'Customer_VPN_Indicator',
    'Customer_flag_terminal_malicious_behavior_1',
    'Customer_flag_terminal_malicious_behavior_2',
    'Customer_flag_terminal_malicious_behavior_3',
    'Customer_flag_terminal_malicious_behavior_4',
    'Customer_flag_terminal_malicious_behavior_5',
    'Customer_flag_terminal_malicious_behavior_6',
    'Customer_inquery_atm_limit',
    'Customer_increase_atm_limit',
    'Account_indicator_release_limit_excess',
    'Account_indicator_Openbanking',
    'Account_release_suspention',
    'Transaction_Failure_Status',
    'Another_Person_Account',
    'Unused_terminal_status',
    'Flag_deposit_more_than_tenMillion',
    'Unused_account_status',
    'Recipient_account_suspend_status',
    'First_time_iOS_by_vulnerable_user',
    'Transaction_is_withdrawal'
]

def phi_coefficient(table):
    table = table.astype(float)

    a = table.loc[0, 0]  # flag=0, normal
    b = table.loc[0, 1]  # flag=0, fraud
    c = table.loc[1, 0]  # flag=1, normal
    d = table.loc[1, 1]  # flag=1, fraud

    denom = np.sqrt((a + b) * (c + d) * (a + c) * (b + d))

    if denom == 0:
        return np.nan

    return ((a * d) - (b * c)) / denom

def odds_ratio_flag1(table):
    """
    flag=1일 때 사기 odds / flag=0일 때 사기 odds
    0 셀 방지를 위해 0.5 보정 적용
    """
    table = table.astype(float) + 0.5

    flag0_normal = table.loc[0, 0]
    flag0_fraud = table.loc[0, 1]
    flag1_normal = table.loc[1, 0]
    flag1_fraud = table.loc[1, 1]

    odds_flag0 = flag0_fraud / flag0_normal
    odds_flag1 = flag1_fraud / flag1_normal

    return odds_flag1 / odds_flag0

def phi_label(phi):
    abs_phi = abs(phi)

    if pd.isna(phi):
        return 'nan'
    elif abs_phi < 0.10:
        return 'negligible'
    elif abs_phi < 0.30:
        return 'small'
    elif abs_phi < 0.50:
        return 'medium'
    else:
        return 'large'

df_flag = train_binary.copy()
df_flag['is_fraud'] = np.where(df_flag[target_col] == normal_label, 0, 1)

results = []

for col in flag_cols:
    temp = df_flag[[col, 'is_fraud']].dropna().copy()

    temp = temp[temp[col].isin([0, 1, '0', '1', True, False])]
    temp[col] = temp[col].astype(int)

    table = pd.crosstab(temp[col], temp['is_fraud'])
    table = table.reindex(index=[0, 1], columns=[0, 1], fill_value=0)

    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)

    phi = phi_coefficient(table)
    odds_ratio = odds_ratio_flag1(table)

    if (row_sums == 0).any() or (col_sums == 0).any():
        results.append({
            'feature': col,
            'test_used': 'not_testable',
            'p_value': np.nan,
            'min_expected_freq': np.nan,
            'phi': phi,
            'abs_phi': abs(phi) if not pd.isna(phi) else np.nan,
            'effect_size': phi_label(phi),
            'odds_ratio_flag1': odds_ratio,
            'reason': 'empty row or column in 2x2 table',
            'n_flag0_normal': table.loc[0, 0],
            'n_flag0_fraud': table.loc[0, 1],
            'n_flag1_normal': table.loc[1, 0],
            'n_flag1_fraud': table.loc[1, 1],
            'n_samples': table.to_numpy().sum()
        })
        continue

    chi2_stat, chi2_p, dof, expected = chi2_contingency(table)
    min_expected = expected.min()

    if min_expected >= 5:
        test_used = 'chi-square'
        p_value = chi2_p
    else:
        test_used = 'fisher_exact'
        _, fisher_p = fisher_exact(table)
        p_value = fisher_p

    results.append({
        'feature': col,
        'test_used': test_used,
        'p_value': p_value,
        'min_expected_freq': min_expected,
        'phi': phi,
        'abs_phi': abs(phi),
        'effect_size': phi_label(phi),
        'odds_ratio_flag1': odds_ratio,
        'reason': '',
        'n_flag0_normal': table.loc[0, 0],
        'n_flag0_fraud': table.loc[0, 1],
        'n_flag1_normal': table.loc[1, 0],
        'n_flag1_fraud': table.loc[1, 1],
        'n_samples': table.to_numpy().sum()
    })

flag_effect_result = (
    pd.DataFrame(results)
    .sort_values('abs_phi', ascending=False)
    .reset_index(drop=True)
)

flag_effect_result

,feature,test_used,p_value,min_expected_freq,phi,abs_phi,effect_size,odds_ratio_flag1,reason,n_flag0_normal,n_flag0_fraud,n_flag1_normal,n_flag1_fraud,n_samples
0,Transaction_is_withdrawal,chi-square,0.000000e+00,358.17,0.142001,0.142001,small,41.375790,,84117,66,34683,1134,120000
1,Flag_deposit_more_than_tenMillion,chi-square,4.359913e-23,509.90,0.028650,0.028650,negligible,1.773748,,68489,521,50311,679,120000
2,Customer_rooting_jailbreak_indicator,chi-square,4.828671e-09,87.93,0.017057,0.017057,negligible,1.700170,,110148,1059,8652,141,120000
3,Account_release_suspention,chi-square,2.800118e-06,432.01,0.013612,0.013612,negligible,1.347065,,42847,354,75953,846,120000
4,Transaction_Failure_Status,chi-square,4.233507e-04,28.12,0.010453,0.010453,negligible,1.727799,,116035,1153,2765,47,120000
5,Customer_flag_terminal_malicious_behavior_2,chi-square,7.510248e-04,115.27,0.009871,0.009871,negligible,1.352673,,107423,1050,11377,150,120000
6,Customer_flag_terminal_malicious_behavior_6,chi-square,1.342738e-03,171.79,0.009376,0.009376,negligible,1.282737,,101832,989,16968,211,120000
7,Customer_VPN_Indicator,chi-square,3.214793e-02,86.41,0.006347,0.006347,negligible,1.257027,,110265,1094,8535,106,120000
8,Customer_increase_atm_limit,chi-square,8.617827e-02,225.40,-0.005061,0.005061,negligible,0.880861,,22291,249,96509,951,120000
9,Customer_inquery_atm_limit,chi-square,1.397791e-01,223.68,-0.004370,0.004370,negligible,0.895280,,22124,244,96676,956,120000
